# Prueba de compatibilidad de la RealSense en Jetson Orin Nano

Abre en una ventana emergente el video capturado en tiempo real por la **Intel RealSense Depth Camera**.

**Flujo**
1. RealSense entrega frames de color (640x480 @ 30 fps, BGR) y frames de profundidad (640x480 @ 30 fps, z16).
2. Se transforma la matriz de profundidad (16 bits) a una imagen visible a color (8 bits).
3. Se muestra en una ventana OpenCV. Tecla `q` para salir.

**Notas**
- Muestra la imagen de **color** en el **lado izquierdo** de la ventana emergente, y la imagen de **profundidad** en el **lado derecho**.

# 1. Importar las librerías + Config de imagen

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 2. Inicialización y ejecución de la RealSense

In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
print("Iniciando pipeline de Intel RealSense... Presiona la tecla 'q' en la ventana para salir.")
pipeline.start(config)

try:
    while True:
        frames = pipeline.wait_for_frames()
        depth_frame = frames.get_depth_frame()
        color_frame = frames.get_color_frame()
        
        # Filtro de seguridad por si se pierde un frame en el USB
        if not depth_frame or not color_frame:
            continue

        depth_image = np.asanyarray(depth_frame.get_data())
        color_image = np.asanyarray(color_frame.get_data())

        # 1. Convertir matriz de profundidad
        depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)

        # 2. Apilar ambas imágenes horizontalmente (Color a la izquierda, Profundidad a la derecha)
        images = np.hstack((color_image, depth_colormap))

        # 3. Renderizar la ventana emergente
        cv2.imshow('Test de Hardware RealSense', images)

        # 4. Refresco de video y gatillo de salida
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    print("Pipeline detenido correctamente.")

Iniciando pipeline de Intel RealSense... Presiona la tecla 'q' en la ventana para salir.
Pipeline detenido correctamente.


## Solución de problemas

| Error | Causa / solución |
|---|---|
| `RuntimeError: No device connected` | RealSense no conectada o sin permisos (udev). |
| Kernel muere al importar | Libs nativas incompatibles o poca RAM; probar el script en terminal. |